# AI-Based Loan Approval System

This notebook demonstrates the complete machine learning pipeline for predicting loan approvals.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Import Libraries

In [ ]:
df = pd.read_csv('dataset/loan_data.csv')
df.head()

## 2. Load Dataset

In [ ]:
print('Dataset Shape:', df.shape)
print('-'*30)
df.info()
print('-'*30)
df.describe()

## 3. Dataset Exploration

In [ ]:
print('Missing Values Before:')
print(df.isnull().sum())

# Handling categorical missing values with Mode
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Loan_Amount_Term', 'Credit_History']:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Handling numerical missing values with Median
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)

print('\nMissing Values After:')
print(df.isnull().sum())

## 4. Missing Value Handling

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)

## 5. Duplicate Checking

In [ ]:
# Drop irrelevant column like Loan_ID
if 'Loan_ID' in df.columns:
    df.drop('Loan_ID', axis=1, inplace=True)

# Encoding categorical variables
le = LabelEncoder()
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

df.head()

## 6. Data Preprocessing & 7. Encoding Categorical Columns

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## 8. Exploratory Data Analysis & 9. Correlation Analysis

In [ ]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training set size:', X_train.shape)
print('Test set size:', X_test.shape)

## 10. Feature Selection & 11. Train-Test Split

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'{name} Accuracy: {acc:.4f}')

## 12. Train Multiple Classification Models

In [ ]:
best_model_name = max(results, key=results.get)
print(f'\nBest Model: {best_model_name}')
best_model = models[best_model_name]

# Detailed Evaluation
y_pred_best = best_model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_best):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_best):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_best):.4f}')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 13. Compare Model Performance & 14. Select Best Model
## 15. Evaluate Best Model

In [ ]:
joblib.dump(best_model, 'models/loan_model.pkl')
print('Model saved to models/loan_model.pkl')

## 16. Save Model using Joblib

In [ ]:
print('Making a prediction example...')
sample = X_test.iloc[[0]]
prediction = best_model.predict(sample)
status = 'Approved' if prediction[0] == 1 else 'Rejected'
print(f'Sample input:\n{sample}')
print(f'Prediction: {status}')

## 17. Prediction Examples